# 01 · Public API data and provenance

Kalshi documents unauthenticated REST access for series, markets and order books. The collector uses GET requests only, bounded retries, source snapshots and request clocks. It has no credential input, portfolio endpoint or order-placement method.

The 90-date universe contains 270 daily events and 1,620 contracts. Historical and live storage tiers must both be queried. Archived candles use different field names from current candles; both encode dollar prices as strings.

In [1]:
from pathlib import Path
import sys, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'kalshi_engine/client.py').exists())
if str(root) not in sys.path: sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
from kalshi_engine.client import ROOT
from kalshi_engine.presentation import show_table, show_figure, equation, read
from kalshi_engine.microstructure import Book, FeePolicy, sweep, execution_fee
from kalshi_engine.arbitrage import partition_audit, temperature_payoff_states, coherence, basket, statewise_portfolio
from kalshi_engine.probability import normalise, calibrate, scores, fit_temperature, paired_block_interval
print('Kalshi Microstructure Research | recorded public-data study | no order placement')

Kalshi Microstructure Research | recorded public-data study | no order placement


In [2]:
metadata = read('data/processed/metadata.json')
candles = read('data/processed/candles.json')
show_table(pd.DataFrame([{'series':s,'historical_contracts':len(v['markets']),'open_contracts':len(v['open_markets']),'fee_type':v['series']['fee_type'],'multiplier':v['series']['fee_multiplier']} for s,v in metadata.items()]), 'Table N01.1. Acquired series and current fee metadata.')
print('Hourly candlesticks:',sum(len(v) for v in candles.values()))
assert len(candles)==1620
assert sum(len(v) for v in candles.values())==56006

series,historical_contracts,open_contracts,fee_type,multiplier
KXHIGHNY,540,12,quadratic,1
KXHIGHCHI,540,12,quadratic,1
KXHIGHMIA,540,12,quadratic,1


Hourly candlesticks: 56006


## Information available at the forecast time

Every accepted feature uses a completed hourly candle ending exactly at the decision timestamp. Final prices, final trading volume, expiration values and winners are not predictors. Labels are stored separately. Candles do not provide historical queue position or executable depth, and a quiet trading interval does not prove that its quotes were fresh.

In [3]:
features = read('data/processed/forecast-features.json')
assert all(all(t==e['cutoff_ts'] for t in e['feature_timestamps']) for e in features)
forbidden={'winner','expiration_value','last_price_dollars','volume_fp','open_interest_fp','result'}
assert all(not forbidden.intersection(e) for e in features)
show_table(pd.read_csv(ROOT/'outputs/tables/coverage.csv').groupby(['hour','status']).size().reset_index(name='events'), 'Table N01.2. Inclusion and missing-hour exclusions.')
print('Temporal alignment and feature-field exclusion checks passed.')

hour,status,events
9,accepted,267
9,missing_exact_completed_hour,3
12,accepted,268
12,missing_exact_completed_hour,2
15,accepted,265
15,missing_exact_completed_hour,5


Temporal alignment and feature-field exclusion checks passed.


## Source integrity and practical failures

The first batch size exceeded the endpoint’s 10,000-candle potential-grid cap. Smaller batches corrected the request without changing the sample. The collector then routed archived markets separately. A concurrent manifest append omitted one index entry, although its complete raw snapshot survived; rebuilding the index from the stored records recovered it. The observed wire hash is an acquisition record; file and canonical-payload hashes independently protect the retained parsed JSON.

In [4]:
import hashlib
index = read('outputs/validation/source-snapshot-index.json')
for row in index:
    assert hashlib.sha256((ROOT/row['file']).read_bytes()).hexdigest()==row['snapshot_sha256']
print('Verified stored source snapshots:',len(index))
print('Distinct live payloads:',read('outputs/supplementary-results.json')['book_payload_hashes'])
print('Raw API data are private research inputs; no public-data licence is inferred.')

Verified stored source snapshots: 521
Distinct live payloads: 20
Raw API data are private research inputs; no public-data licence is inferred.
